# Phase 4 of Logistics Optimization — Basic Vehicle Routing Problem (VRP)

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

# Notebook is inside the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase4_optimization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start


In [2]:
deliveries = pd.read_csv(
    DATA_RAW / "deliveries.csv"
)

riders = pd.read_csv(
    DATA_RAW / "riders.csv"
)

distance_matrix = pd.read_csv(
    DATA_PROCESSED / "network_distance_matrix.csv",
    index_col=0
)

available_riders = riders[
    riders["available"] == True
].copy()

print("Deliveries:", len(deliveries))
print("Available riders:", len(available_riders))
print("Distance matrix shape:", distance_matrix.shape)

Deliveries: 60
Available riders: 5
Distance matrix shape: (61, 61)


In [3]:
location_ids = distance_matrix.index.tolist()

expected_locations = (
    {"DEP001"} |
    set(deliveries["delivery_id"])
)

assert distance_matrix.shape[0] == distance_matrix.shape[1], (
    "Distance matrix must be square."
)

assert distance_matrix.index.is_unique, (
    "Duplicate location IDs found in matrix rows."
)

assert distance_matrix.columns.is_unique, (
    "Duplicate location IDs found in matrix columns."
)

assert set(distance_matrix.index) == expected_locations, (
    "Matrix rows do not match depot and deliveries."
)

assert set(distance_matrix.columns) == expected_locations, (
    "Matrix columns do not match depot and deliveries."
)

# Align columns with row order
distance_matrix = distance_matrix.loc[
    location_ids,
    location_ids
]

assert distance_matrix.notna().all().all(), (
    "Distance matrix contains missing values."
)

assert np.isfinite(distance_matrix.to_numpy()).all(), (
    "Distance matrix contains infinite distances."
)

assert (distance_matrix.to_numpy() >= 0).all(), (
    "Distance matrix contains negative distances."
)

assert np.allclose(
    np.diag(distance_matrix.to_numpy()),
    0
), "Self-distances must be zero."

print("Distance matrix validation passed.")

Distance matrix validation passed.


In [4]:

print("Number of deliveries:", len(deliveries))
print("Available riders:", len(available_riders))

print(
    "Total delivery weight:",
    deliveries["weight_kg"].sum(),
    "kg"
)

print(
    "Combined rider weight capacity:",
    available_riders["capacity_kg"].sum(),
    "kg"
)

print(
    "Combined package capacity:",
    available_riders["max_packages"].sum()
)

display(
    available_riders[
        ["rider_id", "capacity_kg", "max_packages"]
    ]
)


Number of deliveries: 60
Available riders: 5
Total delivery weight: 478.3 kg
Combined rider weight capacity: 460 kg
Combined package capacity: 69


,rider_id,capacity_kg,max_packages
0,R001,100,15
1,R002,100,15
2,R003,80,12
3,R004,100,15
4,R005,80,12


In [5]:

location_ids = distance_matrix.index.tolist()

location_to_index = {
    location_id: index
    for index, location_id in enumerate(location_ids)
}

distance_matrix_int = (
    distance_matrix.round().astype(int)
)

rider_ids = available_riders["rider_id"].tolist()

num_vehicles = len(rider_ids)

depot_index = location_to_index["DEP001"]

print("Locations:", len(location_ids))
print("Vehicles:", num_vehicles)
print("Depot index:", depot_index)


Locations: 61
Vehicles: 5
Depot index: 0


In [6]:

manager = pywrapcp.RoutingIndexManager(
    len(location_ids),
    num_vehicles,
    depot_index
)

routing = pywrapcp.RoutingModel(manager)


In [7]:
print(location_ids)

['DEP001', 'D001', 'D002', 'D003', 'D004', 'D005', 'D006', 'D007', 'D008', 'D009', 'D010', 'D011', 'D012', 'D013', 'D014', 'D015', 'D016', 'D017', 'D018', 'D019', 'D020', 'D021', 'D022', 'D023', 'D024', 'D025', 'D026', 'D027', 'D028', 'D029', 'D030', 'D031', 'D032', 'D033', 'D034', 'D035', 'D036', 'D037', 'D038', 'D039', 'D040', 'D041', 'D042', 'D043', 'D044', 'D045', 'D046', 'D047', 'D048', 'D049', 'D050', 'D051', 'D052', 'D053', 'D054', 'D055', 'D056', 'D057', 'D058', 'D059', 'D060']


In [8]:

def distance_callback(from_index, to_index):

    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)

    from_location = location_ids[from_node]
    to_location = location_ids[to_node]

    return int(
        distance_matrix_int.loc[
            from_location,
            to_location
        ]
    )


distance_callback_index = (
    routing.RegisterTransitCallback(distance_callback)
)

routing.SetArcCostEvaluatorOfAllVehicles(
    distance_callback_index
)


In [9]:

# Use grams to preserve fractional kilogram weights.
WEIGHT_SCALE = 1000

delivery_weights = (
    deliveries.set_index("delivery_id")["weight_kg"]
    .to_dict()
)

weight_demands = {
    "DEP001": 0
}

for delivery_id, weight_kg in delivery_weights.items():
    weight_demands[delivery_id] = int(
        round(float(weight_kg) * WEIGHT_SCALE)
    )

vehicle_weight_capacities = [
    int(round(float(capacity) * WEIGHT_SCALE))
    for capacity in available_riders["capacity_kg"]
]

print("Weight demand example:", list(weight_demands.items())[:5])
print("Vehicle capacities (grams):", vehicle_weight_capacities)


Weight demand example: [('DEP001', 0), ('D001', 2100), ('D002', 5200), ('D003', 9900), ('D004', 4200)]
Vehicle capacities (grams): [100000, 100000, 80000, 100000, 80000]


In [10]:

def weight_callback(from_index):

    node_index = manager.IndexToNode(from_index)

    location_id = location_ids[node_index]

    return weight_demands[location_id]


weight_callback_index = (
    routing.RegisterUnaryTransitCallback(weight_callback)
)

routing.AddDimensionWithVehicleCapacity(
    weight_callback_index,
    0,
    vehicle_weight_capacities,
    True,
    "Weight"
)


True

What the above cell does is that it checks the weight of each delivery and ensures that the total weight assigned to each vehicle does not exceed its capacity. The `weight_callback` function retrieves the weight demand for a given location, and the `AddDimensionWithVehicleCapacity` method adds this constraint to the routing model.

Bear in mind, the weights are scaled to grams to avoid issues with fractional kilograms, which is why we multiply by `WEIGHT_SCALE` (1000) when calculating the weight demands and vehicle capacities. The weight demand is also set to 0 for the depot, as it does not contribute to the weight of deliveries. The weight demand is simply the weight of the package that needs to be delivered to that location.

In [11]:

package_demands = {
    location_id: (
        0 if location_id == "DEP001" else 1
    )
    for location_id in location_ids
}

vehicle_package_capacities = (
    available_riders["max_packages"]
    .astype(int)
    .tolist()
)


def package_callback(from_index):

    node_index = manager.IndexToNode(from_index)

    location_id = location_ids[node_index]

    return package_demands[location_id]


package_callback_index = (
    routing.RegisterUnaryTransitCallback(package_callback)
)

routing.AddDimensionWithVehicleCapacity(
    package_callback_index,
    0,
    vehicle_package_capacities,
    True,
    "Packages"
)


True

What the above cell does is that it checks the number of packages assigned to each vehicle and ensures that it does not exceed the maximum number of packages that the vehicle can carry. The `package_callback` function retrieves the package demand for a given location, and the `AddDimensionWithVehicleCapacity` method adds this constraint to the routing model.

vehicle_package_capacities is a list of the maximum number of packages that each vehicle can carry, which is obtained from the available_riders DataFrame. The package demand is set to 0 for the depot, as it does not contribute to the number of packages being delivered. The package demand is simply 1 for each delivery location, indicating that one package needs to be delivered to that location.

In [12]:
# Allow the optimizer to defer deliveries
# when fleet capacity is insufficient.

UNASSIGNED_PENALTY = 1_000_000

for location_id in location_ids:

    if location_id == "DEP001":
        continue

    node_index = location_to_index[location_id]

    routing_index = manager.NodeToIndex(node_index)

    routing.AddDisjunction(
        [routing_index],
        UNASSIGNED_PENALTY
    )

print("Optional delivery constraints added.")

Optional delivery constraints added.


In [13]:
search_parameters = (
    pywrapcp.DefaultRoutingSearchParameters()
)

search_parameters.first_solution_strategy = (
    routing_enums_pb2
    .FirstSolutionStrategy
    .PARALLEL_CHEAPEST_INSERTION
)

search_parameters.local_search_metaheuristic = (
    routing_enums_pb2
    .LocalSearchMetaheuristic
    .GUIDED_LOCAL_SEARCH
)

search_parameters.time_limit.seconds = 60

solution = routing.SolveWithParameters(
    search_parameters
)

if solution is None:
    raise RuntimeError(
        "No solution found with optional deliveries."
    )

print("Phase 4B solution found.")

Phase 4B solution found.


In [14]:
unassigned_deliveries = []

for location_id in location_ids:

    if location_id == "DEP001":
        continue

    node_index = location_to_index[location_id]

    routing_index = manager.NodeToIndex(node_index)

    if solution.Value(
        routing.NextVar(routing_index)
    ) == routing_index:

        unassigned_deliveries.append(location_id)


unassigned_df = deliveries[
    deliveries["delivery_id"].isin(
        unassigned_deliveries
    )
].copy()

print("Total deliveries:", len(deliveries))

print(
    "Assigned deliveries:",
    len(deliveries) - len(unassigned_deliveries)
)

print(
    "Unassigned deliveries:",
    len(unassigned_deliveries)
)

print(
    "Total deferred weight:",
    round(unassigned_df["weight_kg"].sum(), 2),
    "kg"
)

display(
    unassigned_df[
        ["delivery_id", "weight_kg"]
    ]
)

Total deliveries: 60
Assigned deliveries: 59
Unassigned deliveries: 1
Total deferred weight: 20.5 kg


,delivery_id,weight_kg
35,D036,20.5


In [15]:
route_records = []

for vehicle_id, rider_id in enumerate(rider_ids):

    index = routing.Start(vehicle_id)

    route = []
    total_distance_m = 0

    while not routing.IsEnd(index):

        node_index = manager.IndexToNode(index)
        location_id = location_ids[node_index]

        route.append(location_id)

        next_index = solution.Value(
            routing.NextVar(index)
        )

        next_node = manager.IndexToNode(next_index)
        next_location = location_ids[next_node]

        total_distance_m += int(
            distance_matrix_int.loc[
                location_id,
                next_location
            ]
        )

        index = next_index

    route.append("DEP001")

    assigned_delivery_ids = [
        location_id
        for location_id in route
        if location_id != "DEP001"
    ]

    total_weight_kg = sum(
        weight_demands[delivery_id]
        for delivery_id in assigned_delivery_ids
    ) / WEIGHT_SCALE

    route_records.append({
        "rider_id": rider_id,
        "route": route,
        "delivery_count": len(assigned_delivery_ids),
        "total_weight_kg": total_weight_kg,
        "total_distance_km": total_distance_m / 1000
    })


phase4b_routes_df = pd.DataFrame(route_records)

display(
    phase4b_routes_df[
        [
            "rider_id",
            "delivery_count",
            "total_weight_kg",
            "total_distance_km"
        ]
    ]
)

,rider_id,delivery_count,total_weight_kg,total_distance_km
0,R001,13,99.9,59.516
1,R002,14,98.8,44.618
2,R003,8,79.6,44.587
3,R004,12,100.0,90.033
4,R005,12,79.5,40.127


In [16]:
print("Delivery columns:")
print(deliveries.columns.tolist())

print("\nRider columns:")
print(riders.columns.tolist())

display(deliveries.head(3))
display(available_riders.head(3))

Delivery columns:
['delivery_id', 'customer_id', 'area', 'latitude', 'longitude', 'weight_kg', 'package_size', 'priority', 'window_start', 'window_end', 'service_time_min']

Rider columns:
['rider_id', 'vehicle_type', 'available', 'shift_start', 'shift_end', 'break_start', 'break_end', 'capacity_kg', 'max_packages']


,delivery_id,customer_id,area,latitude,longitude,weight_kg,package_size,priority,window_start,window_end,service_time_min
0,D001,C001,Maryland,6.575974,3.365894,2.1,SMALL,NORMAL,14:00,18:00,15
1,D002,C002,Surulere,6.502082,3.365384,5.2,MEDIUM,NORMAL,12:00,16:00,5
2,D003,C003,Maryland,6.570127,3.365127,9.9,MEDIUM,NORMAL,12:00,16:00,10


,rider_id,vehicle_type,available,shift_start,shift_end,break_start,break_end,capacity_kg,max_packages
0,R001,motorcycle,True,08:00,18:00,13:00,14:00,100,15
1,R002,motorcycle,True,08:00,18:00,13:00,14:00,100,15
2,R003,motorcycle,True,08:00,16:00,12:00,13:00,80,12


In [17]:
from datetime import datetime
import numpy as np
import pandas as pd

required_delivery_cols = {
    "delivery_id",
    "weight_kg",
    "window_start",
    "window_end"
}

required_rider_cols = {
    "rider_id",
    "capacity_kg",
    "max_packages",
    "shift_start",
    "shift_end"
}

missing_delivery = (
    required_delivery_cols - set(deliveries.columns)
)

missing_rider = (
    required_rider_cols - set(available_riders.columns)
)

if missing_delivery or missing_rider:
    raise ValueError(
        f"Missing delivery columns: {missing_delivery}\n"
        f"Missing rider columns: {missing_rider}\n"
        "Define and document scheduling assumptions "
        "before continuing."
    )


def time_to_minutes(value):
    parsed = datetime.strptime(
        str(value).strip(),
        "%H:%M"
    )
    return parsed.hour * 60 + parsed.minute


delivery_windows = {}

for _, row in deliveries.iterrows():
    start = time_to_minutes(row["window_start"])
    end = time_to_minutes(row["window_end"])

    if start > end:
        raise ValueError(
            f"Invalid time window: {row['delivery_id']}"
        )

    delivery_windows[row["delivery_id"]] = (
        start, end
    )


rider_shifts = {}

for _, row in available_riders.iterrows():
    start = time_to_minutes(row["shift_start"])
    end = time_to_minutes(row["shift_end"])

    if start > end:
        raise ValueError(
            f"Invalid rider shift: {row['rider_id']}"
        )

    rider_shifts[row["rider_id"]] = (
        start, end
    )

print("Scheduling data validated.")

Scheduling data validated.


In [18]:
ASSUMED_SPEED_KMH = 25
SERVICE_TIME_MIN = 10

travel_time_matrix = np.ceil(
    (
        distance_matrix.to_numpy(dtype=float)
        / 1000
    )
    / ASSUMED_SPEED_KMH
    * 60
).astype(int)

service_times = {
    location_id: (
        0 if location_id == "DEP001"
        else SERVICE_TIME_MIN
    )
    for location_id in location_ids
}

print(
    "Travel-time matrix shape:",
    travel_time_matrix.shape
)

Travel-time matrix shape: (61, 61)


In [19]:
# Save travel-time matrix
travel_time_df = pd.DataFrame(
    travel_time_matrix,
    index=distance_matrix.index,
    columns=distance_matrix.columns
)

travel_time_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "travel_time_matrix_25kmh.csv"
)

travel_time_path.parent.mkdir(parents=True, exist_ok=True)

travel_time_df.to_csv(travel_time_path)

print("Travel-time matrix saved:", travel_time_path)
print("Matrix shape:", travel_time_df.shape)
print(travel_time_df.head())

Travel-time matrix saved: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\data\processed\travel_time_matrix_25kmh.csv
Matrix shape: (61, 61)
        DEP001  D001  D002  D003  D004  D005  D006  D007  D008  D009  ...  \
DEP001       0    40    18    38    31    25    39     9    28    76  ...   
D001        40     0    24     5    53    23    12    42    49    98  ...   
D002        22    23     0    21    35     8    22    24    32    80  ...   
D003        37     6    20     0    49    20     9    38    46    94  ...   
D004        31    51    30    49     0    31    50    32     5    48  ...   

        D051  D052  D053  D054  D055  D056  D057  D058  D059  D060  
DEP001    16    19    47    41    29    76     6    24    20     6  
D001      30    30    14    15    50    98    40    21    30    39  
D002      12    12    30    24    33    80    22     7    12    21  
D003      26    26    14    15    46   

In [20]:
import numpy as np

np.set_printoptions(threshold=np.inf, linewidth=np.inf)
print(travel_time_matrix)

[[  0  40  18  38  31  25  39   9  28  76  22  53  22  47  31  30  26  20  76  23  14  50  19  36  25  72  24  30  20   3   3  38  36   5  50  28  76  82  21  20   9  17  76  38  37   3  27  47   9  27  29  16  19  47  41  29  76   6  24  20   6]
 [ 40   0  24   5  53  23  12  42  49  98  23  18  22  13  52  52  25  29  98  43  29  15  28   9  23  93  18  51  23  42  38   5  22  37  18  50  98 104  25  28  42  27  98  11  15  40  24  11  42  25  49  30  30  14  15  50  98  40  21  30  39]
 [ 22  23   0  21  35   8  22  24  32  80   5  36   6  30  35  34   9  11  80  26  11  33  10  19   8  76   7  34   5  24  20  21  19  19  33  32  80  86   8  10  24   9  80  22  20  22  10  30  24  10  32  12  12  30  24  33  80  22   7  12  21]
 [ 37   6  20   0  49  20   9  38  46  94  19  19  18  14  48  49  22  26  94  39  26  16  25   5  19  90  15  47  19  39  34   1  18  33  17  46  94 100  22  24  38  23  94   8  12  36  21  12  38  21  46  26  26  14  15  46  94  37  17  26  36]
 [ 31  51  3

In [21]:
manager_c = pywrapcp.RoutingIndexManager(
    len(location_ids),
    num_vehicles,
    depot_index
)

routing_c = pywrapcp.RoutingModel(manager_c)


def distance_callback_c(from_index, to_index):
    from_node = manager_c.IndexToNode(from_index)
    to_node = manager_c.IndexToNode(to_index)

    return int(
        distance_matrix_int.iloc[
            from_node, to_node
        ]
    )


distance_index_c = routing_c.RegisterTransitCallback(
    distance_callback_c
)

routing_c.SetArcCostEvaluatorOfAllVehicles(
    distance_index_c
)


def weight_callback_c(from_index):
    node = manager_c.IndexToNode(from_index)
    return weight_demands[location_ids[node]]


weight_index_c = routing_c.RegisterUnaryTransitCallback(
    weight_callback_c
)

routing_c.AddDimensionWithVehicleCapacity(
    weight_index_c,
    0,
    vehicle_weight_capacities,
    True,
    "Weight"
)


def package_callback_c(from_index):
    node = manager_c.IndexToNode(from_index)
    return package_demands[location_ids[node]]


package_index_c = routing_c.RegisterUnaryTransitCallback(
    package_callback_c
)

routing_c.AddDimensionWithVehicleCapacity(
    package_index_c,
    0,
    vehicle_package_capacities,
    True,
    "Packages"
)

print("Phase 4C routing model initialized.")

Phase 4C routing model initialized.


In [22]:
def time_callback_c(from_index, to_index):
    from_node = manager_c.IndexToNode(from_index)
    to_node = manager_c.IndexToNode(to_index)

    from_location = location_ids[from_node]

    travel_minutes = int(
        travel_time_matrix[from_node, to_node]
    )

    return (
        travel_minutes
        + service_times[from_location]
    )


time_index_c = routing_c.RegisterTransitCallback(
    time_callback_c
)

routing_c.AddDimension(
    time_index_c,
    1440,   # Maximum waiting allowance
    1440,   # Maximum time cumul
    False,  # Start time need not be zero
    "Time"
)

time_dimension_c = routing_c.GetDimensionOrDie(
    "Time"
)

print("Time dimension added.")

Time dimension added.


In [23]:
print(delivery_windows)

{'D001': (840, 1080), 'D002': (720, 960), 'D003': (720, 960), 'D004': (600, 840), 'D005': (600, 840), 'D006': (540, 1080), 'D007': (600, 840), 'D008': (540, 720), 'D009': (540, 720), 'D010': (840, 1080), 'D011': (540, 1080), 'D012': (540, 1080), 'D013': (840, 1080), 'D014': (600, 840), 'D015': (720, 960), 'D016': (540, 720), 'D017': (600, 840), 'D018': (540, 720), 'D019': (540, 720), 'D020': (540, 1080), 'D021': (720, 960), 'D022': (720, 960), 'D023': (540, 1080), 'D024': (840, 1080), 'D025': (540, 1080), 'D026': (720, 960), 'D027': (540, 1080), 'D028': (600, 840), 'D029': (540, 720), 'D030': (540, 720), 'D031': (540, 720), 'D032': (840, 1080), 'D033': (540, 720), 'D034': (540, 1080), 'D035': (720, 960), 'D036': (720, 960), 'D037': (600, 840), 'D038': (840, 1080), 'D039': (720, 960), 'D040': (540, 720), 'D041': (600, 840), 'D042': (540, 720), 'D043': (600, 840), 'D044': (540, 1080), 'D045': (720, 960), 'D046': (600, 840), 'D047': (540, 1080), 'D048': (540, 1080), 'D049': (600, 840), 'D

In [24]:
# Apply delivery time windows
for location_id, (start, end) in delivery_windows.items():
    node = location_to_index[location_id]
    routing_index = manager_c.NodeToIndex(node)

    time_dimension_c.CumulVar(
        routing_index
    ).SetRange(start, end)


# Apply rider shift constraints
for vehicle_id, rider_id in enumerate(rider_ids):
    shift_start, shift_end = rider_shifts[rider_id]

    start_index = routing_c.Start(vehicle_id)
    end_index = routing_c.End(vehicle_id)

    time_dimension_c.CumulVar(
        start_index
    ).SetRange(shift_start, shift_end)

    time_dimension_c.CumulVar(
        end_index
    ).SetRange(shift_start, shift_end)

print("Delivery windows and rider shifts applied.")

Delivery windows and rider shifts applied.


In [25]:
# Phase 4C: Allow deliveries to be deferred

for location_id in location_ids:

    if location_id == "DEP001":
        continue

    node_index = location_to_index[location_id]
    routing_index = manager_c.NodeToIndex(node_index)

    routing_c.AddDisjunction(
        [routing_index],
        UNASSIGNED_PENALTY
    )

print(
    "Phase 4C optional deliveries:",
    routing_c.GetNumberOfDisjunctions()
)

Phase 4C optional deliveries: 60


In [26]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2


# Phase 4C: Configure CVRPTW solver

search_parameters_c = pywrapcp.DefaultRoutingSearchParameters()

search_parameters_c.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
)

search_parameters_c.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)

# Restore a valid time budget
search_parameters_c.time_limit.seconds = 60

# Limit the number of solutions as an additional stopping condition
search_parameters_c.solution_limit = 1000

search_parameters_c.log_search = False

print("Phase 4C search parameters configured.")


Phase 4C search parameters configured.


In [27]:
# Phase 4C: Solve CVRPTW

solution_c = routing_c.SolveWithParameters(
    search_parameters_c
)

if solution_c is not None:
    print("Phase 4C: Feasible solution found!")
    print(f"Objective value: {solution_c.ObjectiveValue():,}")
else:
    print("Phase 4C: No feasible solution found.")

Phase 4C: Feasible solution found!
Objective value: 1,291,341


In [28]:
print("Vehicles:", routing_c.vehicles())
print("Delivery locations:", len(delivery_windows))
print("Optional deliveries:", routing_c.GetNumberOfDisjunctions())

if solution_c is not None:
    print("Solver status: Solution found")
else:
    print("Solver status: No solution found")

Vehicles: 5
Delivery locations: 60
Optional deliveries: 60
Solver status: Solution found


In [29]:
print("Rider shifts:")
for rider_id, (start, end) in rider_shifts.items():
    print(f"{rider_id}: {start} to {end}")

print("\nDelivery time windows:")
for location_id, (start, end) in delivery_windows.items():
    print(f"{location_id}: {start} to {end}")

Rider shifts:
R001: 480 to 1080
R002: 480 to 1080
R003: 480 to 960
R004: 600 to 1140
R005: 540 to 1080

Delivery time windows:
D001: 840 to 1080
D002: 720 to 960
D003: 720 to 960
D004: 600 to 840
D005: 600 to 840
D006: 540 to 1080
D007: 600 to 840
D008: 540 to 720
D009: 540 to 720
D010: 840 to 1080
D011: 540 to 1080
D012: 540 to 1080
D013: 840 to 1080
D014: 600 to 840
D015: 720 to 960
D016: 540 to 720
D017: 600 to 840
D018: 540 to 720
D019: 540 to 720
D020: 540 to 1080
D021: 720 to 960
D022: 720 to 960
D023: 540 to 1080
D024: 840 to 1080
D025: 540 to 1080
D026: 720 to 960
D027: 540 to 1080
D028: 600 to 840
D029: 540 to 720
D030: 540 to 720
D031: 540 to 720
D032: 840 to 1080
D033: 540 to 720
D034: 540 to 1080
D035: 720 to 960
D036: 720 to 960
D037: 600 to 840
D038: 840 to 1080
D039: 720 to 960
D040: 540 to 720
D041: 600 to 840
D042: 540 to 720
D043: 600 to 840
D044: 540 to 1080
D045: 720 to 960
D046: 600 to 840
D047: 540 to 1080
D048: 540 to 1080
D049: 600 to 840
D050: 720 to 960
D051: 

In [30]:
# Phase 4C: Extract optimized routes and delivery metrics

if solution_c is None:
    raise RuntimeError("Phase 4C has no solution to evaluate.")

phase4c_route_records = []
phase4c_stop_records = []

for vehicle_id, rider_id in enumerate(rider_ids):

    index = routing_c.Start(vehicle_id)

    route = []
    route_distance_m = 0
    delivered_weight_g = 0
    delivered_packages = 0

    start_time = solution_c.Value(
        time_dimension_c.CumulVar(index)
    )

    while not routing_c.IsEnd(index):

        node = manager_c.IndexToNode(index)
        location_id = location_ids[node]

        arrival_time = solution_c.Value(
            time_dimension_c.CumulVar(index)
        )

        route.append(location_id)

        if location_id != "DEP001":
            delivered_weight_g += weight_demands[location_id]
            delivered_packages += 1

            phase4c_stop_records.append({
                "rider_id": rider_id,
                "delivery_id": location_id,
                "arrival_min": arrival_time,
                "window_start": delivery_windows[location_id][0],
                "window_end": delivery_windows[location_id][1]
            })

        next_index = solution_c.Value(
            routing_c.NextVar(index)
        )

        next_node = manager_c.IndexToNode(next_index)

        route_distance_m += int(
            distance_matrix_int.iloc[node, next_node]
        )

        index = next_index

    end_time = solution_c.Value(
        time_dimension_c.CumulVar(index)
    )

    route.append("DEP001")

    phase4c_route_records.append({
        "rider_id": rider_id,
        "route": " -> ".join(route),
        "deliveries": delivered_packages,
        "weight_kg": delivered_weight_g / WEIGHT_SCALE,
        "distance_km": route_distance_m / 1000,
        "start_min": start_time,
        "end_min": end_time,
        "duration_min": end_time - start_time
    })

phase4c_routes_df = pd.DataFrame(phase4c_route_records)
phase4c_stops_df = pd.DataFrame(phase4c_stop_records)

display(phase4c_routes_df)

,rider_id,route,deliveries,weight_kg,distance_km,start_min,end_min,duration_min
0,R001,DEP001 -> D033 -> D052 -> D059 -> D017 -> D028...,14,100.0,63.195,480,953,473
1,R002,DEP001 -> D044 -> D054 -> D053 -> D003 -> D023...,9,99.3,54.481,480,898,418
2,R003,DEP001 -> D029 -> D048 -> D040 -> D007 -> D019...,12,79.5,42.375,480,805,325
3,R004,DEP001 -> D060 -> D041 -> D031 -> D026 -> D012...,12,99.2,45.867,600,886,286
4,R005,DEP001 -> D030 -> D057 -> D056 -> D009 -> D037...,12,79.8,85.423,540,870,330


In [31]:
# Phase 4C: Identify deferred deliveries

deferred_ids_c = []

for location_id in delivery_windows:

    node = location_to_index[location_id]
    index = manager_c.NodeToIndex(node)

    if solution_c.Value(routing_c.NextVar(index)) == index:
        deferred_ids_c.append(location_id)

phase4c_deferred_df = deliveries[
    deliveries["delivery_id"].isin(deferred_ids_c)
].copy()

total_deliveries = len(deliveries)
completed_deliveries = total_deliveries - len(deferred_ids_c)

print("PHASE 4C PERFORMANCE SUMMARY")
print("-" * 40)

print("Total deliveries:", total_deliveries)
print("Completed deliveries:", completed_deliveries)
print("Deferred deliveries:", len(deferred_ids_c))

print(
    "Delivery completion rate:",
    f"{completed_deliveries / total_deliveries * 100:.2f}%"
)

print(
    "Total distance:",
    f"{phase4c_routes_df['distance_km'].sum():.2f} km"
)

print(
    "Deferred weight:",
    f"{phase4c_deferred_df['weight_kg'].sum():.2f} kg"
)

display(phase4c_deferred_df)

PHASE 4C PERFORMANCE SUMMARY
----------------------------------------
Total deliveries: 60
Completed deliveries: 59
Deferred deliveries: 1
Delivery completion rate: 98.33%
Total distance: 291.34 km
Deferred weight: 20.50 kg


,delivery_id,customer_id,area,latitude,longitude,weight_kg,package_size,priority,window_start,window_end,service_time_min
35,D036,C036,Ajah,6.471891,3.577304,20.5,LARGE,NORMAL,12:00,16:00,7


In [32]:
# Phase 4C: Validate operational constraints

time_window_violations = phase4c_stops_df[
    (phase4c_stops_df["arrival_min"] <
     phase4c_stops_df["window_start"])
    |
    (phase4c_stops_df["arrival_min"] >
     phase4c_stops_df["window_end"])
]

capacity_violations = []

shift_violations = []

for vehicle_id, rider_id in enumerate(rider_ids):

    row = phase4c_routes_df[
        phase4c_routes_df["rider_id"] == rider_id
    ].iloc[0]

    if (
        row["weight_kg"] >
        vehicle_weight_capacities[vehicle_id] / WEIGHT_SCALE
        or
        row["deliveries"] >
        vehicle_package_capacities[vehicle_id]
    ):
        capacity_violations.append(rider_id)

    shift_start, shift_end = rider_shifts[rider_id]

    if (
        row["start_min"] < shift_start
        or row["end_min"] > shift_end
    ):
        shift_violations.append(rider_id)

print("PHASE 4C VALIDATION")
print("-" * 40)

print("Time-window violations:", len(time_window_violations))
print("Capacity violations:", len(capacity_violations))
print("Shift violations:", len(shift_violations))

if (
    len(time_window_violations) == 0
    and len(capacity_violations) == 0
    and len(shift_violations) == 0
):
    print("\nAll checked operational constraints satisfied!")
else:
    print("\nSome constraints require investigation.")

PHASE 4C VALIDATION
----------------------------------------
Time-window violations: 0
Capacity violations: 0
Shift violations: 0

All checked operational constraints satisfied!


In [33]:

# Phase 4C: Record reproducibility baseline

import ortools
import platform

phase4c_baseline = {
    "completed_deliveries": completed_deliveries,
    "deferred_deliveries": len(deferred_ids_c),
    "total_distance_km": round(
        phase4c_routes_df["distance_km"].sum(), 3
    ),
    "objective_value": solution_c.ObjectiveValue(),
    "deferred_ids": sorted(deferred_ids_c),
    "ortools_version": ortools.__version__,
    "python_version": platform.python_version(),
    "search_time_limit_seconds": (
        search_parameters_c.time_limit.seconds
    )
}

print("PHASE 4C BASELINE")
print("-" * 40)

for key, value in phase4c_baseline.items():
    print(f"{key}: {value}")


PHASE 4C BASELINE
----------------------------------------
completed_deliveries: 59
deferred_deliveries: 1
total_distance_km: 291.341
objective_value: 1291341
deferred_ids: ['D036']
ortools_version: 9.15.6755
python_version: 3.14.0
search_time_limit_seconds: 60


In [34]:

# Phase 4C: Preserve selected baseline for later phases

import json

BASELINE_DIR = OUTPUT_DIR / "selected_baseline"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

phase4c_routes_df.to_csv(
    BASELINE_DIR / "phase4c_routes.csv",
    index=False
)

phase4c_stops_df.to_csv(
    BASELINE_DIR / "phase4c_stops.csv",
    index=False
)

phase4c_deferred_df.to_csv(
    BASELINE_DIR / "phase4c_deferred.csv",
    index=False
)

with open(
    BASELINE_DIR / "phase4c_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        phase4c_baseline,
        f,
        indent=4
    )

print("Selected Phase 4C baseline saved.")
print("Location:", BASELINE_DIR)


Selected Phase 4C baseline saved.
Location: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\outputs\phase4_optimization\selected_baseline


# Phase 4D — Business-Aware Routing Optimization

## Step 4D.1: Inspect delivery priorities

Objective:
Understand how deliveries are classified by business priority
before designing priority-aware deferral penalties.

In [35]:

# Phase 4D.1: Inspect delivery priorities

print("DELIVERY PRIORITY DISTRIBUTION")
print("-" * 40)

print(deliveries["priority"].value_counts(dropna=False))

print("\nPRIORITY CATEGORIES")
print("-" * 40)

print(deliveries["priority"].unique())

print("\nSAMPLE DELIVERIES")
print("-" * 40)

display(
    deliveries[
        [
            "delivery_id",
            "priority",
            "weight_kg",
            "window_start",
            "window_end"
        ]
    ].head(10)
)


DELIVERY PRIORITY DISTRIBUTION
----------------------------------------
priority
NORMAL    28
HIGH      20
LOW       10
URGENT     2
Name: count, dtype: int64

PRIORITY CATEGORIES
----------------------------------------
<StringArray>
['NORMAL', 'HIGH', 'LOW', 'URGENT']
Length: 4, dtype: str

SAMPLE DELIVERIES
----------------------------------------


,delivery_id,priority,weight_kg,window_start,window_end
0,D001,NORMAL,2.1,14:00,18:00
1,D002,NORMAL,5.2,12:00,16:00
2,D003,NORMAL,9.9,12:00,16:00
3,D004,HIGH,4.2,10:00,14:00
4,D005,NORMAL,5.7,10:00,14:00
5,D006,NORMAL,19.3,09:00,18:00
6,D007,HIGH,3.5,10:00,14:00
7,D008,NORMAL,1.9,09:00,12:00
8,D009,HIGH,2.8,09:00,12:00
9,D010,NORMAL,4.7,14:00,18:00


In [36]:

# Phase 4D.1B: Define business-priority penalties

PRIORITY_PENALTIES = {
    "URGENT": 5_000_000,
    "HIGH": 3_000_000,
    "NORMAL": 2_000_000,
    "LOW": 1_000_000
}

# Check that every delivery has a valid priority
priority_values = (
    deliveries["priority"]
    .astype(str)
    .str.strip()
    .str.upper()
)

unknown_priorities = set(priority_values) - set(PRIORITY_PENALTIES)

assert not unknown_priorities, (
    f"Unknown priority categories: {unknown_priorities}"
)

# Assign each delivery its penalty
deliveries = deliveries.copy()

deliveries["deferral_penalty"] = priority_values.map(
    PRIORITY_PENALTIES
)

print("PRIORITY PENALTY DISTRIBUTION")
print("-" * 40)

print(
    deliveries.groupby("priority")["deferral_penalty"]
    .agg(["count", "min", "max"])
)

print("\nSAMPLE PENALTIES")
display(
    deliveries[
        ["delivery_id", "priority", "deferral_penalty"]
    ].head(10)
)


PRIORITY PENALTY DISTRIBUTION
----------------------------------------
          count      min      max
priority                         
HIGH         20  3000000  3000000
LOW          10  1000000  1000000
NORMAL       28  2000000  2000000
URGENT        2  5000000  5000000

SAMPLE PENALTIES


,delivery_id,priority,deferral_penalty
0,D001,NORMAL,2000000
1,D002,NORMAL,2000000
2,D003,NORMAL,2000000
3,D004,HIGH,3000000
4,D005,NORMAL,2000000
5,D006,NORMAL,2000000
6,D007,HIGH,3000000
7,D008,NORMAL,2000000
8,D009,HIGH,3000000
9,D010,NORMAL,2000000


In [37]:
# Phase 4D.2: Prepare operational data

required_delivery_cols_d = {
    "delivery_id", "priority", "service_time_min",
    "weight_kg", "window_start", "window_end",
    "deferral_penalty"
}

required_rider_cols_d = {
    "rider_id", "break_start", "break_end"
}

assert required_delivery_cols_d.issubset(deliveries.columns), (
    f"Missing delivery columns: "
    f"{required_delivery_cols_d - set(deliveries.columns)}"
)

assert required_rider_cols_d.issubset(available_riders.columns), (
    f"Missing rider columns: "
    f"{required_rider_cols_d - set(available_riders.columns)}"
)

assert deliveries["delivery_id"].is_unique
assert deliveries["service_time_min"].notna().all()
assert (deliveries["service_time_min"] >= 0).all()
assert deliveries["deferral_penalty"].notna().all()

service_times_d = {"DEP001": 0}

for _, row in deliveries.iterrows():
    service_times_d[row["delivery_id"]] = int(
        np.ceil(float(row["service_time_min"]))
    )

penalty_by_delivery_d = (
    deliveries.set_index("delivery_id")["deferral_penalty"]
    .astype(int)
    .to_dict()
)

# Interpret break_start and break_end as the allowable
# window in which a break must be completed.
# A 30-minute duration is an experimental assumption.
BREAK_DURATION_MIN = 30

rider_breaks_d = {}

for _, row in available_riders.iterrows():
    start = time_to_minutes(row["break_start"])
    end = time_to_minutes(row["break_end"])

    if end - start < BREAK_DURATION_MIN:
        raise ValueError(
            f"Insufficient break window for {row['rider_id']}"
        )

    rider_breaks_d[row["rider_id"]] = (start, end)

print("Phase 4D operational data validated.")
print("Service times:", len(service_times_d))
print("Priority penalties:", len(penalty_by_delivery_d))
print("Rider break windows:", rider_breaks_d)


Phase 4D operational data validated.
Service times: 61
Priority penalties: 60
Rider break windows: {'R001': (780, 840), 'R002': (780, 840), 'R003': (720, 780), 'R004': (840, 900), 'R005': (780, 840)}


In [38]:
# Phase 4D.3: Initialize routing model

manager_d = pywrapcp.RoutingIndexManager(
    len(location_ids),
    num_vehicles,
    depot_index
)

routing_d = pywrapcp.RoutingModel(manager_d)

def distance_callback_d(from_index, to_index):
    from_node = manager_d.IndexToNode(from_index)
    to_node = manager_d.IndexToNode(to_index)

    return int(
        distance_matrix_int.iloc[from_node, to_node]
    )

distance_index_d = routing_d.RegisterTransitCallback(
    distance_callback_d
)

routing_d.SetArcCostEvaluatorOfAllVehicles(
    distance_index_d
)

print("Phase 4D routing model initialized.")


Phase 4D routing model initialized.


In [39]:
# Phase 4D.4: Weight and package constraints

def weight_callback_d(from_index):
    node = manager_d.IndexToNode(from_index)
    return weight_demands[location_ids[node]]

weight_index_d = routing_d.RegisterUnaryTransitCallback(
    weight_callback_d
)

routing_d.AddDimensionWithVehicleCapacity(
    weight_index_d,
    0,
    vehicle_weight_capacities,
    True,
    "Weight"
)

def package_callback_d(from_index):
    node = manager_d.IndexToNode(from_index)
    return package_demands[location_ids[node]]

package_index_d = routing_d.RegisterUnaryTransitCallback(
    package_callback_d
)

routing_d.AddDimensionWithVehicleCapacity(
    package_index_d,
    0,
    vehicle_package_capacities,
    True,
    "Packages"
)

print("Weight and package constraints added.")


Weight and package constraints added.


In [40]:
# Phase 4D.5: Time dimension and scheduling constraints

def time_callback_d(from_index, to_index):
    from_node = manager_d.IndexToNode(from_index)
    to_node = manager_d.IndexToNode(to_index)

    from_location = location_ids[from_node]

    travel_minutes = int(
        travel_time_matrix[from_node, to_node]
    )

    return (
        travel_minutes
        + service_times_d[from_location]
    )

time_index_d = routing_d.RegisterTransitCallback(
    time_callback_d
)

routing_d.AddDimension(
    time_index_d,
    1440,
    1440,
    False,
    "Time"
)

time_dimension_d = routing_d.GetDimensionOrDie("Time")

# Customer time windows
for delivery_id, (start, end) in delivery_windows.items():
    node = location_to_index[delivery_id]
    index = manager_d.NodeToIndex(node)

    time_dimension_d.CumulVar(index).SetRange(
        start, end
    )

# Rider shifts
for vehicle_id, rider_id in enumerate(rider_ids):
    shift_start, shift_end = rider_shifts[rider_id]

    time_dimension_d.CumulVar(
        routing_d.Start(vehicle_id)
    ).SetRange(shift_start, shift_end)

    time_dimension_d.CumulVar(
        routing_d.End(vehicle_id)
    ).SetRange(shift_start, shift_end)

print("Delivery time windows and shifts applied.")


Delivery time windows and shifts applied.


In [41]:
# Phase 4D.6: Rider break intervals

solver_d = routing_d.solver()

# Service duration associated with each routing index
visit_transits_d = []

for index in range(routing_d.Size()):
    node = manager_d.IndexToNode(index)
    location_id = location_ids[node]

    visit_transits_d.append(
        service_times_d[location_id]
    )

break_intervals_d = {}

for vehicle_id, rider_id in enumerate(rider_ids):
    break_start, break_end = rider_breaks_d[rider_id]

    break_interval = solver_d.FixedDurationIntervalVar(
        break_start,
        break_end - BREAK_DURATION_MIN,
        BREAK_DURATION_MIN,
        False,
        f"Break_{rider_id}"
    )

    time_dimension_d.SetBreakIntervalsOfVehicle(
        [break_interval],
        vehicle_id,
        visit_transits_d
    )

    break_intervals_d[rider_id] = break_interval

print("Mandatory rider break intervals added.")


Mandatory rider break intervals added.


In [42]:
# Phase 4D.7: Priority-weighted optional deliveries

for delivery_id in location_ids:
    if delivery_id == "DEP001":
        continue

    node = location_to_index[delivery_id]
    index = manager_d.NodeToIndex(node)

    routing_d.AddDisjunction(
        [index],
        penalty_by_delivery_d[delivery_id]
    )

print(
    "Priority-weighted optional deliveries:",
    routing_d.GetNumberOfDisjunctions()
)


Priority-weighted optional deliveries: 60


In [43]:
# Phase 4D.8: Workload balancing

# Penalize the longest route duration.
# Experimental value; tune during evaluation.
WORKLOAD_SPAN_COEFFICIENT = 10

time_dimension_d.SetGlobalSpanCostCoefficient(
    WORKLOAD_SPAN_COEFFICIENT
)

print("Workload balancing objective added.")


Workload balancing objective added.


In [44]:
# Phase 4D.9: Solve business-aware CVRPTW

search_parameters_d = pywrapcp.DefaultRoutingSearchParameters()

search_parameters_d.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy
    .PARALLEL_CHEAPEST_INSERTION
)

search_parameters_d.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic
    .GUIDED_LOCAL_SEARCH
)

search_parameters_d.time_limit.seconds = 60
search_parameters_d.log_search = False

solution_d = routing_d.SolveWithParameters(
    search_parameters_d
)

if solution_d is None:
    raise RuntimeError(
        "Phase 4D returned no solution. "
        "Investigate break feasibility and solver configuration."
    )

print("PHASE 4D SOLUTION FOUND")
print("-" * 40)
print("Objective:", solution_d.ObjectiveValue())


PHASE 4D SOLUTION FOUND
----------------------------------------
Objective: 2386995


In [45]:
# Phase 4D.10: Extract rider routes and stops

phase4d_route_records = []
phase4d_stop_records = []

for vehicle_id, rider_id in enumerate(rider_ids):
    index = routing_d.Start(vehicle_id)

    route = []
    route_distance_m = 0
    delivered_weight_g = 0
    delivered_packages = 0

    start_time = solution_d.Value(
        time_dimension_d.CumulVar(index)
    )

    while not routing_d.IsEnd(index):
        node = manager_d.IndexToNode(index)
        delivery_id = location_ids[node]

        route.append(delivery_id)

        arrival = solution_d.Value(
            time_dimension_d.CumulVar(index)
        )

        if delivery_id != "DEP001":
            delivered_weight_g += weight_demands[delivery_id]
            delivered_packages += 1

            phase4d_stop_records.append({
                "rider_id": rider_id,
                "delivery_id": delivery_id,
                "priority": deliveries.set_index(
                    "delivery_id"
                ).at[delivery_id, "priority"],
                "arrival_min": arrival,
                "window_start": delivery_windows[delivery_id][0],
                "window_end": delivery_windows[delivery_id][1],
                "service_time_min": service_times_d[delivery_id]
            })

        next_index = solution_d.Value(
            routing_d.NextVar(index)
        )

        next_node = manager_d.IndexToNode(next_index)

        route_distance_m += int(
            distance_matrix_int.iloc[node, next_node]
        )

        index = next_index

    end_time = solution_d.Value(
        time_dimension_d.CumulVar(index)
    )

    route.append("DEP001")

    phase4d_route_records.append({
        "rider_id": rider_id,
        "route": " -> ".join(route),
        "deliveries": delivered_packages,
        "weight_kg": delivered_weight_g / WEIGHT_SCALE,
        "distance_km": route_distance_m / 1000,
        "start_min": start_time,
        "end_min": end_time,
        "duration_min": end_time - start_time
    })

phase4d_routes_df = pd.DataFrame(phase4d_route_records)
phase4d_stops_df = pd.DataFrame(phase4d_stop_records)

display(phase4d_routes_df)


,rider_id,route,deliveries,weight_kg,distance_km,start_min,end_min,duration_min
0,R001,DEP001 -> D030 -> D052 -> D017 -> D041 -> D019...,13,99.9,117.973,556,1015,459
1,R002,DEP001 -> D033 -> D008 -> D056 -> D009 -> D018...,12,99.2,86.355,538,1015,477
2,R003,DEP001 -> D053 -> D011 -> D034 -> D031 -> D003...,10,76.5,54.966,605,960,355
3,R004,DEP001 -> D057 -> D060 -> D007 -> D040 -> D048...,15,100.0,75.965,612,953,341
4,R005,DEP001 -> D029 -> D016 -> D012 -> D054 -> D032...,8,80.0,46.966,679,929,250


In [46]:
# Phase 4D.11: Deferred deliveries

deferred_ids_d = []

for delivery_id in location_ids:
    if delivery_id == "DEP001":
        continue

    node = location_to_index[delivery_id]
    index = manager_d.NodeToIndex(node)

    if solution_d.Value(routing_d.NextVar(index)) == index:
        deferred_ids_d.append(delivery_id)

phase4d_deferred_df = deliveries[
    deliveries["delivery_id"].isin(deferred_ids_d)
].copy()

completed_d = len(deliveries) - len(deferred_ids_d)

print("PHASE 4D PERFORMANCE")
print("-" * 40)

print("Total deliveries:", len(deliveries))
print("Completed deliveries:", completed_d)
print("Deferred deliveries:", len(deferred_ids_d))
print("Completion rate:", f"{100 * completed_d / len(deliveries):.2f}%")
print("Distance:", round(phase4d_routes_df["distance_km"].sum(), 3), "km")
print("Objective:", solution_d.ObjectiveValue())

print("\nDEFERRED DELIVERIES")
display(
    phase4d_deferred_df[
        ["delivery_id", "priority", "weight_kg", "deferral_penalty"]
    ]
)

print("\nCOMPLETED DELIVERIES BY PRIORITY")
completed_priority_d = deliveries[
    ~deliveries["delivery_id"].isin(deferred_ids_d)
]["priority"].value_counts()

display(completed_priority_d)


PHASE 4D PERFORMANCE
----------------------------------------
Total deliveries: 60
Completed deliveries: 58
Deferred deliveries: 2
Completion rate: 96.67%
Distance: 382.225 km
Objective: 2386995

DEFERRED DELIVERIES


,delivery_id,priority,weight_kg,deferral_penalty
36,D037,LOW,13.6,1000000
41,D042,LOW,9.1,1000000



COMPLETED DELIVERIES BY PRIORITY


priority
NORMAL    28
HIGH      20
LOW        8
URGENT     2
Name: count, dtype: int64

In [47]:
# Phase 4D.12: Operational validation

time_violations_d = phase4d_stops_df[
    (phase4d_stops_df["arrival_min"] <
     phase4d_stops_df["window_start"])
    |
    (phase4d_stops_df["arrival_min"] >
     phase4d_stops_df["window_end"])
]

capacity_violations_d = []
shift_violations_d = []

for vehicle_id, rider_id in enumerate(rider_ids):
    row = phase4d_routes_df[
        phase4d_routes_df["rider_id"] == rider_id
    ].iloc[0]

    if (
        row["weight_kg"] >
        vehicle_weight_capacities[vehicle_id] / WEIGHT_SCALE
        or row["deliveries"] >
        vehicle_package_capacities[vehicle_id]
    ):
        capacity_violations_d.append(rider_id)

    shift_start, shift_end = rider_shifts[rider_id]

    if (
        row["start_min"] < shift_start
        or row["end_min"] > shift_end
    ):
        shift_violations_d.append(rider_id)

# Verify delivery accounting
assigned_ids_d = phase4d_stops_df["delivery_id"].tolist()

assert len(assigned_ids_d) == len(set(assigned_ids_d)), (
    "Duplicate delivery assignment detected."
)

assert set(assigned_ids_d).isdisjoint(deferred_ids_d)

assert set(assigned_ids_d) | set(deferred_ids_d) == set(
    deliveries["delivery_id"]
)

print("PHASE 4D VALIDATION")
print("-" * 40)
print("Time-window violations:", len(time_violations_d))
print("Capacity violations:", len(capacity_violations_d))
print("Shift violations:", len(shift_violations_d))
print("Delivery accounting: PASSED")

assert len(time_violations_d) == 0
assert len(capacity_violations_d) == 0
assert len(shift_violations_d) == 0

print("\nAll independently checked constraints satisfied.")
print("Break feasibility is enforced by the routing model.")


PHASE 4D VALIDATION
----------------------------------------
Time-window violations: 0
Capacity violations: 0
Shift violations: 0
Delivery accounting: PASSED

All independently checked constraints satisfied.
Break feasibility is enforced by the routing model.


In [48]:
# Phase 4D.13: Compare operational performance

comparison_df = pd.DataFrame([
    {
        "Model": "Phase 4C",
        "Completed": len(deliveries) - len(deferred_ids_c),
        "Deferred": len(deferred_ids_c),
        "Completion %": round(
            100 * (len(deliveries) - len(deferred_ids_c))
            / len(deliveries), 2
        ),
        "Distance km": round(
            phase4c_routes_df["distance_km"].sum(), 3
        ),
        "Longest rider duration min": (
            phase4c_routes_df["duration_min"].max()
        ),
        "Duration spread min": (
            phase4c_routes_df["duration_min"].max()
            - phase4c_routes_df["duration_min"].min()
        )
    },
    {
        "Model": "Phase 4D",
        "Completed": completed_d,
        "Deferred": len(deferred_ids_d),
        "Completion %": round(
            100 * completed_d / len(deliveries), 2
        ),
        "Distance km": round(
            phase4d_routes_df["distance_km"].sum(), 3
        ),
        "Longest rider duration min": (
            phase4d_routes_df["duration_min"].max()
        ),
        "Duration spread min": (
            phase4d_routes_df["duration_min"].max()
            - phase4d_routes_df["duration_min"].min()
        )
    }
])

display(comparison_df)

print("\nDEFERRED PRIORITY COMPARISON")

priority_comparison = pd.DataFrame({
    "Phase 4C": deliveries[
        deliveries["delivery_id"].isin(deferred_ids_c)
    ]["priority"].value_counts(),
    "Phase 4D": phase4d_deferred_df[
        "priority"
    ].value_counts()
}).fillna(0).astype(int)

display(priority_comparison)


,Model,Completed,Deferred,Completion %,Distance km,Longest rider duration min,Duration spread min
0,Phase 4C,59,1,98.33,291.341,473,187
1,Phase 4D,58,2,96.67,382.225,477,227



DEFERRED PRIORITY COMPARISON


,Phase 4C,Phase 4D
priority,,
LOW,0,2
NORMAL,1,0


In [49]:
# Phase 4D.14: Export results

phase4d_routes_df.to_csv(
    OUTPUT_DIR / "phase4d_routes.csv",
    index=False
)

phase4d_stops_df.to_csv(
    OUTPUT_DIR / "phase4d_stops.csv",
    index=False
)

phase4d_deferred_df.to_csv(
    OUTPUT_DIR / "phase4d_deferred.csv",
    index=False
)

comparison_df.to_csv(
    OUTPUT_DIR / "phase4c_vs_phase4d.csv",
    index=False
)

print("Phase 4D results exported successfully.")


Phase 4D results exported successfully.


In [50]:
# PHASE 4E: FINAL BASELINE AUDIT
# Run after Phase 4D.14 in the same notebook.

from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the exported results to verify that the files exist and are readable.
files = {
    "Phase 4D routes": OUTPUT_DIR / "phase4d_routes.csv",
    "Phase 4D stops": OUTPUT_DIR / "phase4d_stops.csv",
    "Phase 4D deferred": OUTPUT_DIR / "phase4d_deferred.csv",
    "Phase 4C vs 4D comparison": OUTPUT_DIR / "phase4c_vs_phase4d.csv",
}

results = {}

print("=" * 70)
print("PHASE 4 FINAL BASELINE AUDIT")
print("=" * 70)

for name, path in files.items():
    print(f"\n{name}")
    print("-" * 70)

    if not path.exists():
        print(f"FILE MISSING: {path}")
        continue

    df = pd.read_csv(path)
    results[name] = df

    print(f"File: {path}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")

    if df.empty:
        print("WARNING: This file contains no rows.")
    else:
        display(df.head(5))

# Summarize the Phase 4C vs Phase 4D comparison.
comparison = results.get("Phase 4C vs 4D comparison")

if comparison is not None:
    print("\n" + "=" * 70)
    print("PHASE 4C VS PHASE 4D COMPARISON")
    print("=" * 70)

    display(comparison)

# Record the exact schema and row counts for reproducibility.
audit_rows = []

for name, path in files.items():
    if path.exists():
        df = results.get(name)
        audit_rows.append({
            "dataset": name,
            "file": str(path),
            "rows": len(df) if df is not None else None,
            "columns": ", ".join(df.columns) if df is not None else None,
        })

audit_df = pd.DataFrame(audit_rows)
audit_path = OUTPUT_DIR / "phase4_final_audit.csv"
audit_df.to_csv(audit_path, index=False)

print("\n" + "=" * 70)
print(f"Audit inventory saved to: {audit_path}")
print("This confirms file availability and structure, not constraint validity.")

PHASE 4 FINAL BASELINE AUDIT

Phase 4D routes
----------------------------------------------------------------------
File: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\outputs\phase4_optimization\phase4d_routes.csv
Rows: 5
Columns: ['rider_id', 'route', 'deliveries', 'weight_kg', 'distance_km', 'start_min', 'end_min', 'duration_min']


,rider_id,route,deliveries,weight_kg,distance_km,start_min,end_min,duration_min
0,R001,DEP001 -> D030 -> D052 -> D017 -> D041 -> D019...,13,99.9,117.973,556,1015,459
1,R002,DEP001 -> D033 -> D008 -> D056 -> D009 -> D018...,12,99.2,86.355,538,1015,477
2,R003,DEP001 -> D053 -> D011 -> D034 -> D031 -> D003...,10,76.5,54.966,605,960,355
3,R004,DEP001 -> D057 -> D060 -> D007 -> D040 -> D048...,15,100.0,75.965,612,953,341
4,R005,DEP001 -> D029 -> D016 -> D012 -> D054 -> D032...,8,80.0,46.966,679,929,250



Phase 4D stops
----------------------------------------------------------------------
File: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\outputs\phase4_optimization\phase4d_stops.csv
Rows: 58
Columns: ['rider_id', 'delivery_id', 'priority', 'arrival_min', 'window_start', 'window_end', 'service_time_min']


,rider_id,delivery_id,priority,arrival_min,window_start,window_end,service_time_min
0,R001,D030,NORMAL,559,540,720,15
1,R001,D052,NORMAL,592,540,720,7
2,R001,D017,NORMAL,600,600,840,15
3,R001,D041,HIGH,621,600,840,5
4,R001,D019,NORMAL,649,540,720,12



Phase 4D deferred
----------------------------------------------------------------------
File: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\outputs\phase4_optimization\phase4d_deferred.csv
Rows: 2
Columns: ['delivery_id', 'customer_id', 'area', 'latitude', 'longitude', 'weight_kg', 'package_size', 'priority', 'window_start', 'window_end', 'service_time_min', 'deferral_penalty']


,delivery_id,customer_id,area,latitude,longitude,weight_kg,package_size,priority,window_start,window_end,service_time_min,deferral_penalty
0,D037,C037,Lekki,6.469513,3.597717,13.6,LARGE,LOW,10:00,14:00,12,1000000
1,D042,C042,Ajah,6.480823,3.560854,9.1,MEDIUM,LOW,09:00,12:00,15,1000000



Phase 4C vs 4D comparison
----------------------------------------------------------------------
File: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\outputs\phase4_optimization\phase4c_vs_phase4d.csv
Rows: 2
Columns: ['Model', 'Completed', 'Deferred', 'Completion %', 'Distance km', 'Longest rider duration min', 'Duration spread min']


,Model,Completed,Deferred,Completion %,Distance km,Longest rider duration min,Duration spread min
0,Phase 4C,59,1,98.33,291.341,473,187
1,Phase 4D,58,2,96.67,382.225,477,227



PHASE 4C VS PHASE 4D COMPARISON


,Model,Completed,Deferred,Completion %,Distance km,Longest rider duration min,Duration spread min
0,Phase 4C,59,1,98.33,291.341,473,187
1,Phase 4D,58,2,96.67,382.225,477,227



Audit inventory saved to: C:\Users\ginik\Documents\CODING\Portfolio-projects\Project-2-African-Logistics-Intelligence\african-logistics-intelligence-start\outputs\phase4_optimization\phase4_final_audit.csv
This confirms file availability and structure, not constraint validity.
